# Surrogate MGGP + PSO — Ackley 2D
**Surrogate-assisted optimization**: MGGP fits a surrogate on 300 LHS samples,
then PSO minimizes on the surrogate surface.

Figures produced:
- `fig03_surrogate_obs_pred.png` — Surrogate quality: Observed × Predicted
- `fig03_surrogate_surface.png` — 3D: true Ackley surface vs MGGP surrogate
- `fig03_pso_ackley_contour.png` — PSO spatial distribution: true vs surrogate
- `fig03_pso_convergence_surrogate.png` — PSO convergence on the surrogate


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pathlib import Path
from sklearn.metrics import r2_score
import os, math

FIG_DIR = Path("figures")
os.makedirs(FIG_DIR, exist_ok=True)

TRAIN_COLOR  = '#1f77b4'
VAL_COLOR    = '#ff7f0e'
TEST_COLOR   = '#2ca02c'
RUN_A_COLOR  = '#1f77b4'
RUN_B_COLOR  = '#d62728'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.linestyle': '-', 'grid.alpha': 0.4,
    'grid.color': '#cccccc', 'font.size': 11, 'axes.labelsize': 12,
    'axes.titlesize': 13, 'legend.fontsize': 10,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})

def obs_pred_ax(ax, y_tr, yp_tr, y_v, yp_v, y_te, yp_te, title,
                xlabel="Observed", ylabel="Predicted"):
    h1 = ax.scatter(y_tr, yp_tr, c=TRAIN_COLOR, marker='o', s=40, alpha=0.75,
                    label=f'Train  (R²={r2_score(y_tr, yp_tr):.6f})')
    h2 = ax.scatter(y_v,  yp_v,  c=VAL_COLOR,   marker='s', s=40, alpha=0.75,
                    label=f'Val    (R²={r2_score(y_v,  yp_v):.6f})')
    h3 = ax.scatter(y_te, yp_te, c=TEST_COLOR,  marker='^', s=40, alpha=0.75,
                    label=f'Test   (R²={r2_score(y_te, yp_te):.6f})')
    all_y = np.concatenate([y_tr, y_v, y_te])
    lo, hi = all_y.min(), all_y.max()
    ax.plot([lo, hi], [lo, hi], color='black', lw=1.5)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    return h1, h2, h3

print("Style loaded.")

def _snap(ax, log_y=False):
    ax.figure.canvas.draw()
    ax.tick_params(top=True, right=True, which='both', direction='in')
    lo, hi = ax.get_xlim()
    xt = sorted(t for t in ax.get_xticks() if lo - 1e-9 <= t <= hi + 1e-9)
    if len(xt) >= 2:
        ax.set_xlim(xt[0], xt[-1])
    if log_y:
        lo, hi = ax.get_ylim()
        if lo > 0 and hi > 0:
            lo_dec = 10 ** math.floor(math.log10(lo))
            hi_log = math.log10(hi)
            frac   = hi_log - math.floor(hi_log)
            hi_dec = 10 ** (math.floor(hi_log) if frac < 0.02 else math.ceil(hi_log))
            if lo_dec < hi_dec:
                ax.set_ylim(lo_dec, hi_dec)
    else:
        lo, hi = ax.get_ylim()
        yt = sorted(t for t in ax.get_yticks() if lo - 1e-9 <= t <= hi + 1e-9)
        if len(yt) >= 2:
            ax.set_ylim(yt[0], yt[-1])

def _fix_cbar(cb, cp):
    _lo, _hi = cp.get_clim()
    _t = [x for x in cb.get_ticks() if _lo < x < _hi]
    cb.set_ticks([_lo] + _t + [_hi])


## Data generation (70/15/15 split from 300 LHS samples)

In [3]:
from symgene.benchmarks import ackley_2d

bench = ackley_2d()
rng   = np.random.default_rng(0)

def lhs(rng, lo, hi, n):
    d = len(lo)
    X = np.empty((n, d))
    for j in range(d):
        perm = rng.permutation(n)
        X[:, j] = lo[j] + (perm + rng.uniform(0, 1, n)) / n * (hi[j] - lo[j])
    return X

lo, hi = np.array([-5., -5.]), np.array([5., 5.])
X_all  = lhs(rng, lo, hi, 400)
y_all  = np.array([bench.fn(X_all[i]) for i in range(400)])

n_train, n_val = 280, 60            # 70 / 15 / 15
X_train = X_all[:n_train];           y_train = y_all[:n_train]
X_val   = X_all[n_train:n_train+n_val]; y_val = y_all[n_train:n_train+n_val]
X_test  = X_all[n_train+n_val:];     y_test  = y_all[n_train+n_val:]

print(f"Train:{X_train.shape[0]}  Val:{X_val.shape[0]}  Test:{X_test.shape[0]}")
print(f"y range: [{y_all.min():.3f}, {y_all.max():.3f}]")

Train:280  Val:60  Test:60
y range: [0.581, 14.302]


## MGGP Surrogate

| Parameter | Value | Description |
|---|---|---|
| `n_genes` | 4 | Fixed gene count |
| `pop_size` | 60 | Population size |
| `n_gen` | 80 | Generations |
| `regression_degree` | 1 | Linear combination of gene outputs |
| `combiner` | `ridge` | Ridge regularisation |


In [4]:
from symgene import SymGeneRegressor
from symgene.primitives import STANDARD
from symgene.selection import TournamentSelection
from symgene.metrics import mae, r2, mape
from symgene.optimization import PSOOptimizer

reg = SymGeneRegressor(
    n_genes=4, pop_size=80, n_gen=200,
    primitives=STANDARD, squash={"lim": 8, "alpha": 0.1, "scale": 2.0},
    feature_names=["x1", "x2"], seed=0, verbose=0,
    combiner="ridge",
    ridge_alphas=[0.01, 0.1, 1.0, 10.0, 100.0],
    regression_degree=1,
    selection=TournamentSelection(size=4),
    mutpb=0.25, mutpb_low=0.30,
    mutation_weights=[0.5, 1.5, 1.0],
)
reg.fit(X_train, y_train, X_val=X_val, y_val=y_val)
print("Surrogate fitted.")
print(f"Genes     : {reg.n_genes_}")
print(f"Expression: {reg.best_expression_[:120]}...")

Surrogate fitted.
Genes     : 30
Expression: sigmoid(sqrt(log(relu(min2(sqrt(log(relu(min2(cos(x1), inv(sqrt(abs(sub(cos(x1), inv(log(x1)))))))))), inv(sqrt(abs(sub(...


## PSO on the Surrogate


In [5]:
optimizer = PSOOptimizer(n_particles=50, n_iter=300, verbose=0)

pso_res = optimizer.optimize(
    lambda x: float(reg.predict(x.reshape(1, -1))[0]),
    bounds=bench.bounds, seed=0,
)
f_true = float(bench.fn(pso_res.x_best))

print(f"PSO found: x={np.round(pso_res.x_best, 4)}  f_surrogate={pso_res.f_best:.4f}  f_true={f_true:.6f}")
print(f"True optimum: f={bench.f_opt} at x={bench.x_opt}")

PSO found: x=[ 0.0453 -0.    ]  f_surrogate=0.3753  f_true=0.181745
True optimum: f=0.0 at x=[0. 0.]


## Figure 1 — Surrogate Quality: Observed × Predicted


In [ ]:
yp_tr = reg.predict(X_train)
yp_v = reg.predict(X_val)
yp_te = reg.predict(X_test)

fig, ax = plt.subplots(1, 1, figsize=(7, 6))

obs_pred_ax(
    ax,
    y_train,
    yp_tr,
    y_val,
    yp_v,
    y_test,
    yp_te,
    title="MGGP Surrogate  (Observed × Predicted)",
    xlabel="Observed Ackley f(x)",
    ylabel="Surrogate Prediction",
)

# 1. Remove extra borders from plotted points
for collection in ax.collections:
    collection.set_linewidth(0)
    collection.set_edgecolor("none")

for line in ax.lines:
    if line.get_marker() != "None" and line.get_marker() != "":
        line.set_linewidth(0)
        line.set_markeredgewidth(0)

# 2. Apply _snap
_snap(ax)

# 3. Add grid
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6, color="gray")

# 4. Ensure solid black axes, labels and borders
ax.tick_params(colors="black", which="both")
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.0)

# 5. Legend with clean icons
leg = ax.legend(
    loc="upper left",
    frameon=True,
    edgecolor="black",
    fancybox=False,
    fontsize=9,
    handlelength=1.2,
    handletextpad=0.5,
)

for handle in leg.legend_handles:
    if hasattr(handle, "set_linewidth"):
        handle.set_linewidth(0)

plt.savefig(
    os.path.join(FIG_DIR, "fig03_surrogate_obs_pred.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()
print("Saved: fig03_surrogate_obs_pred.png")

## Figure 2 — 3D Surface: True Ackley vs MGGP Surrogate
Left: true Ackley 2D (cmap `viridis`). Right: MGGP surrogate (cmap `plasma`).


In [ ]:
def ackley_vec(x1, x2):
    return (
        -20 * np.exp(-0.2 * np.sqrt(0.5 * (x1**2 + x2**2)))
        - np.exp(0.5 * (np.cos(2 * np.pi * x1) + np.cos(2 * np.pi * x2)))
        + np.e
        + 20
    )


# Grid for both surfaces
x1g = np.linspace(-5, 5, 70)
x2g = np.linspace(-5, 5, 70)
X1g, X2g = np.meshgrid(x1g, x2g)
X_grid = np.column_stack([X1g.ravel(), X2g.ravel()])

Z_true = ackley_vec(X1g, X2g)
yp_test = reg.predict(X_test)
Z_surr = reg.predict(X_grid).reshape(X1g.shape)

r2_val = r2_score(y_test, yp_test)

ELEV, AZIM = 28, -55

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5.2),
    subplot_kw={"projection": "3d"},
    gridspec_kw={"wspace": -0.3},
)

plt.suptitle(
    "Ackley 2D — True Surface vs MGGP Surrogate",
    fontsize=13,
    y=0.98,
)

panels = [
    (Z_true, "True Function", "viridis"),
    (Z_surr, f"MGGP Surrogate\nR² (test) = {r2_val:.8f}", "plasma"),
]

ticks_xy = [-5, -2.5, 0, 2.5, 5]

# --- Custom tick labels with space padding for the corner ---
# Add trailing spaces after '5' on X1 to push the label left
x_labels = ["-5", "-2.5", "0", "2.5", "5   "]

# Add leading spaces before '-5' on X2 to push the label right
y_labels = ["   -5", "-2.5", "0", "2.5", "5"]

z_min_val = 0.0
z_max_val = float(np.ceil(max(Z_true.max(), Z_surr.max())))

for ax, (Z, title, cmap) in zip(axes, panels):
    ax.computed_zorder = False

    ax.plot_surface(
        X1g,
        X2g,
        Z,
        cmap=cmap,
        alpha=0.92,
        linewidth=0,
        antialiased=True,
    )

    ax.set_title(title, fontsize=11, pad=-2)

    ax.set_xlabel("$x_1$", color="black", labelpad=4)
    ax.set_ylabel("$x_2$", color="black", labelpad=4)

    ax.set_xticks(ticks_xy)
    ax.set_xticklabels(x_labels)

    ax.set_yticks(ticks_xy)
    ax.set_yticklabels(y_labels)

    ax.zaxis.set_rotate_label(False)
    ax.set_zlabel(
        "$f(x_1,x_2)$",
        color="black",
        fontsize=9,
        labelpad=1,
        rotation=90,
        ha="right",
        va="bottom",
    )

    ax.set_zlim(z_min_val, z_max_val)
    ax.set_zticks(np.linspace(z_min_val, z_max_val, 5))

    ax.set_box_aspect(None, zoom=0.9)

    ax.view_init(elev=ELEV, azim=AZIM)
    ax.tick_params(colors="black", which="both", pad=3)

    if "_snap" in globals():
        _snap(ax)

plt.subplots_adjust(left=0.01, right=0.97, top=0.85, bottom=0.03)

output_path = os.path.join(FIG_DIR, "fig03_surrogate_surface.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0.2)
plt.show()
print(f"Saved: {output_path}")

## Figure 3 — PSO Spatial Distribution: True vs Surrogate
Two panels: true Ackley landscape (log scale, cmap `viridis`) and MGGP surrogate (cmap `plasma`).
Black star = true optimum at (0, 0). Red diamond = best point found by PSO on the surrogate.


In [ ]:
x1g_c = np.linspace(-5, 5, 300)
x2g_c = np.linspace(-5, 5, 300)
X1g_c, X2g_c = np.meshgrid(x1g_c, x2g_c)
X_mesh = np.column_stack([X1g_c.ravel(), X2g_c.ravel()])

Z_true_c = ackley_vec(X1g_c, X2g_c)
Z_surr_c = reg.predict(X_mesh).reshape(X1g_c.shape)

# Apply log1p before determining the scale
Z_true_log = np.log1p(np.clip(Z_true_c, 0, None))
Z_surr_log = np.log1p(np.clip(Z_surr_c, 0, None))

# --- Fixed and evenly spaced colorscale configuration ---
V_MIN = 0.0
# Set ceiling to the data maximum (Ackley log1p ≈ 2.6–3.0)
V_MAX = float(max(np.max(Z_true_log), np.max(Z_surr_log)))

# 35 uniform levels for the gradient
levels = np.linspace(V_MIN, V_MAX, 35)

# 6 evenly spaced ticks from base to top (e.g. 0.0, 0.6, 1.2, 1.8, 2.4, 3.0)
cb_ticks = np.linspace(V_MIN, V_MAX, 6)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plt.subplots_adjust(wspace=0.35)

panels = [
    (Z_true_log, "Ackley 2D — True Function", "viridis", None, None),
    (Z_surr_log, "MGGP Surrogate", "plasma", pso_res, f_true),
]

for ax, (Z_log, title, cmap, pso, ft) in zip(axes, panels):
    # Force contour to respect exactly V_MIN and V_MAX
    cp = ax.contourf(
        X1g_c,
        X2g_c,
        Z_log,
        levels=levels,
        vmin=V_MIN,
        vmax=V_MAX,
        cmap=cmap,
        alpha=0.88,
    )

    # Colorbar locked to regular linspace ticks
    _cb = plt.colorbar(
        cp, ax=ax, label="log(1 + f)", shrink=0.85, ticks=cb_ticks
    )
    _fix_cbar(_cb, cp)

    ax.scatter(
        *bench.x_opt,
        color="black",
        marker="*",
        s=70,
        zorder=6,
        label=f"True optimum  f={bench.f_opt}",
    )
    if pso is not None:
        ax.scatter(
            pso.x_best[0],
            pso.x_best[1],
            color="red",
            marker="D",
            s=15,
            zorder=7,
            edgecolors="black",
            lw=0.8,
            label=f"PSO found  f_true={ft:.2e}",
        )

    ax.set_xlabel("$x_1$", color="black")
    ax.set_ylabel("$x_2$", color="black")
    ax.set_title(title, color="black")
    ax.legend(
        loc="upper right",
        frameon=True,
        edgecolor="black",
        fancybox=False,
        fontsize=9,
    )

    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)

    # Apply _snap before restoring borders and grid
    if "_snap" in globals():
        _snap(ax)

    # Restore grid and standardise black borders
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5, color="gray")
    ax.tick_params(colors="black", which="both")
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("black")
        spine.set_linewidth(1.0)

plt.savefig(
    os.path.join(FIG_DIR, "fig03_pso_ackley_contour.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()
print("Saved: fig03_pso_ackley_contour.png")

## Figure 4 — PSO Convergence on the Surrogate Surface

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6))

ax.semilogy(
    pso_res.history,
    color=RUN_A_COLOR,
    lw=2,
    label="MGGP Surrogate",
)

# Axis labels and title
ax.set_xlabel("Iteration", color="black", labelpad=5)
ax.set_ylabel("Best fitness on surrogate (log scale)", color="black", labelpad=5)
ax.set_title("PSO Convergence on MGGP Surrogate — Ackley 2D", fontsize=11, pad=8)

# 1. Apply _snap
if "_snap" in globals():
    _snap(ax, log_y=True)

# 2. Add grid (major and minor lines for log scale)
ax.grid(True, which="major", linestyle="--", linewidth=0.6, alpha=0.7, color="gray")
ax.grid(True, which="minor", linestyle=":", linewidth=0.4, alpha=0.4, color="gray")

# 3. Ensure solid black axes, labels and borders
ax.tick_params(colors="black", which="both")
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.0)

# 4. Clean legend aligned to the exact style
leg = ax.legend(
    loc="upper right",
    frameon=True,
    edgecolor="black",
    fancybox=False,
    fontsize=9,
    handlelength=1.2,
    handletextpad=0.5,
)

for handle in leg.legend_handles:
    if hasattr(handle, "set_linewidth"):
        handle.set_linewidth(1.5)

# 5. Save high-resolution output
output_path = os.path.join(FIG_DIR, "fig03_pso_convergence_surrogate.png")
plt.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {output_path}")

## Numerical Results

In [10]:
print("=== MGGP Surrogate ===")
print(f"  Surrogate R²   (test) : {r2(y_test,   yp_te):.4f}")
print(f"  Surrogate MAE  (test) : {mae(y_test,  yp_te):.4f}")
print(f"  Surrogate MAPE (test) : {mape(y_test, yp_te):.2f}%")
print(f"  PSO found x           : {np.round(pso_res.x_best, 6)}")
print(f"  Surrogate f(x*)       : {pso_res.f_best:.6f}")
print(f"  True f(x*)            : {f_true:.6f}")
print(f"  Gap to optimum        : {abs(f_true - bench.f_opt):.6f}")
print()
print(f"True optimum : f={bench.f_opt}  at x={bench.x_opt}")

=== MGGP Surrogate ===
  Surrogate R²   (test) : 0.9738
  Surrogate MAE  (test) : 0.2588
  Surrogate MAPE (test) : 2.60%
  PSO found x           : [ 0.045279 -0.      ]
  Surrogate f(x*)       : 0.375257
  True f(x*)            : 0.181745
  Gap to optimum        : 0.181745

True optimum : f=0.0  at x=[0. 0.]
